# ResearchPilot — Multi-Agent AI Research & Decision System

**Final Project Walkthrough | AI agents, retrieval, verification, orchestration, and evaluation**

This notebook is a guided walkthrough of the **updated ResearchPilot codebase**. It is designed to help me understand every stage, demonstrate the system, and explain the AI logic in the final face-camera video.

## 1. The problem this project solves

A normal chatbot can produce a convincing answer even when its evidence is missing or unrelated. ResearchPilot is designed to answer a research question using **retrieved passages from uploaded PDFs**, preserve where each passage came from, analyze claims, check citations, and build a report that separates verified findings from missing evidence.

The main idea is **evidence first, answer second**:

1. Receive a research question and any selected documents.
2. Plan the work and retrieve relevant evidence.
3. Let the language model propose structured findings tied to source IDs.
4. Validate the sources and claims with programmatic checks and a review step.
5. Generate and store a Markdown report from the verification results.
6. Abstain from a final recommendation when the required evidence is not verified.

## How to run this notebook

- Open it in VS Code from the **updated `researchpilot/` project folder** and select the project's `.venv` Python kernel.
- Install the project dependencies once with `uv sync --all-groups`, then restart the kernel.
- Run cells from top to bottom. The first model load downloads `Qwen/Qwen2.5-1.5B-Instruct` if it is not cached; an Internet connection and several gigabytes of free storage may be required.
- Model timing and generated wording vary by machine, package version, and cached model. Treat displayed results as actual run outputs, not fixed promises.

> **Compatibility note:** these walkthrough cells target the updated `app/` layout (for example, `app/rag/qa.py` and `app/services/llm.py`). The separately supplied older starter ZIP uses a different `src/researchpilot/` layout and is not interchangeable with this notebook without a code migration.

## What this notebook proves — and what it does not

It demonstrates real PDF extraction, embeddings, FAISS retrieval, a local Qwen model, the actual FastAPI routes and LangGraph workflow, report generation, an evidence-absent question, and automated checks. Demo web search may use **synthetic sample pages**; those must never be described as live web evidence. The notebook also documents limitations rather than claiming the system is error-free.

## 2. Architecture: the end-to-end workflow

```text
Research question + selected PDF IDs
                 │
                 ▼
          Planner (Python)
                 │
       ┌─────────┴─────────┐
       ▼                   ▼
 Web search / Demo     PDF document search
       └─────────┬─────────┘
                 ▼
           Collect sources
                 ▼
          Assess evidence
           ┌─────┴─────┐
     Not enough       Enough
           │             │
           ▼             ▼
   Evidence-limited   Analyst (Qwen)
       report             ▼
                      Verifier
                         │
            at most one bounded revision
                         │
                         ▼
                   Report Writer
                         │
                         ▼
                Saved run + Markdown report
```

Search and PDF retrieval can run in parallel. `assess_evidence` decides whether enough evidence exists to continue. The graph can stop early, generate a limited report, or perform the analysis-and-verification path. Rejected findings can trigger **one revision at most**, preventing an unbounded loop.

LangGraph makes the nodes and routes explicit; it is not the language model itself. The Planner is deterministic Python. MiniLM supplies retrieval embeddings. Qwen proposes findings and can review claim support. Ordinary Python validation and report code handle important decisions outside the model.

## 3. Environment and device selection

Before testing the AI behavior, the notebook checks that it is running from the correct project directory, verifies required libraries, and chooses an available compute device.

The project setup command is `uv sync --all-groups`. The device-selection cell only selects CUDA, Apple MPS, or CPU; **it does not load Qwen yet**. Actual model loading happens in Section 6.

In [ ]:
import os
import sys
from pathlib import Path


def find_project_root() -> Path:
    # Find the repository from the kernel's working directory.
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "app" / "rag" / "qa.py").exists() and (folder / "pyproject.toml").exists():
            return folder
    raise FileNotFoundError("Open this notebook from inside the researchpilot/ project folder.")


PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Python:      ", sys.executable)
if Path(sys.prefix).resolve() != (PROJECT_ROOT / ".venv").resolve():
    print("WARNING: this kernel is not the project's .venv.")
    print("In VS Code: Select Kernel → Python Environments → .venv")

### Cell explanation — locate the project and prepare imports

- `os`, `sys`, and `Path` are standard Python tools for directories and import configuration.
- `find_project_root()` checks the current working directory and its parents. It accepts a folder only when both the expected RAG file and `pyproject.toml` exist.
- `os.chdir(PROJECT_ROOT)` makes relative paths start from the repository root.
- `sys.path.insert(0, ...)` lets the notebook import the local `app` package.
- `sys.executable` shows which Python is running. `sys.prefix` is compared with `.venv` to warn about the wrong environment.

**What to say (English):** “This cell prepares the notebook environment and locates the project. It does not load an AI model; it makes sure the later imports come from the correct project and Python environment.”

**بالمصري:** الخلية دي بتظبط مكان التشغيل والـ Python environment عشان الـ imports تشتغل صح. لسه مفيش موديل AI اتحمّل.

In [ ]:
import importlib.util
from importlib.metadata import version

required = {
    "fastapi": "fastapi",
    "langgraph": "langgraph",
    "pypdf": "pypdf",
    "faiss": "faiss-cpu",
    "sentence_transformers": "sentence-transformers",
    "torch": "torch",
    "transformers": "transformers",
}
missing = [pkg for module, pkg in required.items() if importlib.util.find_spec(module) is None]
if missing:
    raise ImportError(f"Missing {', '.join(missing)}. Run `uv sync --all-groups`, then restart.")
print("  ".join(f"{pkg} {version(pkg)}" for pkg in required.values()))

### Cell explanation — confirm dependencies

- `required` maps each import name to the installed package name. For example, the import is `faiss`, while the distribution installed is `faiss-cpu`.
- `importlib.util.find_spec(...)` checks whether the import can be found without importing and running the package.
- If anything is missing, the cell raises a clear error and tells you to install the project groups.
- The final line prints installed package versions, which helps diagnose environment differences.

This cell **checks** dependencies; it does not install them automatically.

**What to say (English):** “I verify that the libraries needed for the API, graph, PDF parsing, vector search, and model are installed before running the pipeline.”

**بالمصري:** بنراجع إن المكتبات الأساسية موجودة وإصداراتها إيه، ولو حاجة ناقصة الكود بيقولنا نثبت dependencies بدل ما نكتشف المشكلة في نص التشغيل.

In [ ]:
import torch

from app.services.llm import select_device

# Prefer CUDA, then Apple MPS, then CPU; float16 keeps the model near 3.1 GB.
device_config = select_device("auto")
DEVICE = device_config.device
print(f"CUDA: {torch.cuda.is_available()}   MPS: {torch.backends.mps.is_available()}")
print(f"Using {device_config.device} ({device_config.dtype}): {device_config.reason}")

### Cell explanation — choose the compute device

- `torch` reports whether CUDA or Apple's Metal Performance Shaders (MPS) is available.
- `select_device("auto")` uses the project's own selection logic to choose CUDA, Apple MPS, or CPU and the suitable data type.
- `DEVICE` is passed to the LLM loader later. It is only a choice at this stage; it does not load model weights.
- GPU selection can improve speed, but actual speed depends on the machine and model.

**What to say (English):** “The code selects the best available device. On a supported Mac it can use Apple MPS; otherwise it can use CUDA or fall back to CPU.”

**بالمصري:** هنا بنختار الموديل هيشتغل على أنهي جهاز، لكن تحميل Qwen نفسه بيحصل في خلية تانية.

## 4. PDF ingestion: preserve the evidence location

The example document is a **synthetic five-page battery-pilot report**, created for testing. It is not a real-world battery study. Using a controlled PDF lets us inspect whether the system retrieves the correct passages and refuses questions that the PDF cannot answer.

`load_pdf()` extracts text one page at a time. Keeping the page boundary means every later chunk can preserve the page number used in citations. The `document_id` is a stable content hash, so it identifies the file without asking the model to invent an ID. The table displays each page's text length and opening text.

In [ ]:
import json

from IPython.display import Markdown, display

from app.rag.pdf import load_pdf

SAMPLE_DIR = PROJECT_ROOT / "sample_data" / "pdfs"
CASES = json.loads((SAMPLE_DIR / "researchpilot_test_report_cases.json").read_text())
SAMPLE_PDF = SAMPLE_DIR / CASES["pdf"]
document = load_pdf(SAMPLE_PDF)


def table(headers, rows):
    lines = ["| " + " | ".join(headers) + " |", "|" + "---|" * len(headers)]
    lines += [
        "| " + " | ".join(str(v).replace("|", "\\|").replace("\n", " ") for v in row) + " |"
        for row in rows
    ]
    display(Markdown("\n".join(lines)))


print(f"{document.filename}: {document.page_count} pages, document_id {document.document_id}")
table(
    ["Page", "Characters", "Start of page"],
    [(p.page_number, len(p.text), p.text[:90] + "…") for p in document.pages],
)

### Cell explanation — inspect the PDF extraction

- The cell loads the JSON test cases so the demo PDF and questions stay linked to one test fixture.
- `load_pdf(SAMPLE_PDF)` extracts each page separately and returns a structured document object.
- The `table()` helper formats rows as a Markdown table for display; it does not change document content.
- The table shows each page number, character count, and the start of extracted text. Empty or unexpectedly short pages can reveal extraction problems.
- `document_id` is read from the loader, not invented by the model.

**Important:** this report is synthetic test data. It is useful for repeatable testing, but it is not an authoritative real battery report.

**What to say (English):** “I extract the PDF page by page and preserve the page metadata, because later citations must point back to the actual page.”

**بالمصري:** بنقرأ كل صفحة لوحدها عشان لما النظام يطلع معلومة نعرف رجعها من صفحة كام، وده مهم في الـ citations.

## 5. Chunking, embeddings, and FAISS retrieval

A whole PDF may be too large and unfocused for one model prompt, so ResearchPilot divides the text into smaller passages.

- **Chunking:** about 500 characters per chunk with 100 characters of overlap. Chunks do not cross PDF pages, allowing page-level citation metadata to survive.
- **Embedding model:** `sentence-transformers/all-MiniLM-L6-v2`, which maps each passage or query to a numeric vector with 384 dimensions.
- **FAISS:** stores/searches the passage vectors and returns nearest matches.
- **Retrieval threshold:** `VectorRetriever(..., min_score=0.25)` filters low-scoring matches when the retriever is used by the QA pipeline. The small search demonstration below calls `store.search()` directly to display the top four results, so it is an illustrative nearest-neighbor view rather than a thresholded final answer.

A score is a similarity signal, not proof that a passage answers the question. That is why the system has later verification stages.

In [ ]:
from app.rag.chunking import chunk_document
from app.rag.embeddings import SentenceTransformerEmbeddings
from app.rag.retrieval import VectorRetriever
from app.rag.vector_store import FaissVectorStore

chunks = chunk_document(document)
embedder = SentenceTransformerEmbeddings()
vectors = embedder.embed_documents([c.text for c in chunks])
store = FaissVectorStore(embedder.dimension)
store.add(chunks, vectors)
retriever = VectorRetriever(embedder, store, min_score=0.25)

print(f"{len(chunks)} chunks, embedding matrix {vectors.shape}")
query = "Which battery option was selected and why?"
hits = store.search(embedder.embed_query(query), top_k=4)
print(f"Query: {query}")
table(
    ["Rank", "Similarity", "Page", "Chunk ID", "Text"],
    [
        (h.rank, f"{h.score:.3f}", h.chunk.page_number, h.chunk.chunk_id, h.chunk.text[:80] + "…")
        for h in hits
    ],
)

### Cell explanation — create the searchable vector index

- `chunk_document(document)` splits the extracted pages into smaller passages with page-aware IDs.
- `SentenceTransformerEmbeddings()` loads the embedding model wrapper; `embed_documents(...)` maps every chunk to a numeric vector.
- `FaissVectorStore(embedder.dimension)` creates a vector index with the embedding dimension expected by the model.
- `store.add(...)` indexes the chunks and their vectors.
- `VectorRetriever(..., min_score=0.25)` wraps the store with a minimum similarity threshold for the QA pipeline.
- The demo query is embedded using `embed_query(...)`; `store.search(..., top_k=4)` displays the four nearest chunks.

The printed shape is approximately `(number_of_chunks, embedding_dimension)`. A vector distance/similarity ranks candidates; it does not by itself establish factual support.

**What to say (English):** “The PDF text becomes overlapping chunks, each chunk becomes an embedding vector, and FAISS lets us retrieve the passages closest in meaning to the user’s question.”

**بالمصري:** بنحوّل أجزاء الـ PDF لأرقام تمثل المعنى، وFAISS بيجيب أقرب الأجزاء للسؤال. ده استرجاع للمعلومة، مش إثبات إن الإجابة صح.

## 6. Load the local LLM: Qwen

The analysis model is `Qwen/Qwen2.5-1.5B-Instruct`, loaded through the project's `HuggingFaceLLM` wrapper. The notebook reuses the same instance for the later standalone QA and API workflow.

The wrapper loads lazily: constructing the object does not by itself load the model weights. `llm.load()` performs the real load and raises a clear error if it fails, rather than silently substituting pretend output. The tokenizer example shows that text becomes sub-word token IDs; models process tokens, not whole words as indivisible units.

This project does not fine-tune Qwen in this notebook. It uses the pretrained instruction model with the configured generation strategy.

In [ ]:
import time

from app.services.llm import HuggingFaceLLM, LLMLoadError

llm = HuggingFaceLLM("Qwen/Qwen2.5-1.5B-Instruct", device=DEVICE)
started = time.perf_counter()
try:
    llm.load()
except LLMLoadError as exc:
    raise RuntimeError(
        f"{exc}\n\nThe first run downloads about 3.1 GB and needs Internet."
    ) from None
print(f"Loaded in {time.perf_counter() - started:.1f}s:", llm.describe())

ids = llm.tokenizer("Lithium iron phosphate batteries.")["input_ids"]
print("Tokens:", llm.tokenizer.convert_ids_to_tokens(ids))

### Cell explanation — load Qwen and inspect tokenization

- `HuggingFaceLLM(...)` configures the model ID and selected device.
- `time.perf_counter()` measures elapsed load time.
- `llm.load()` downloads/loads the tokenizer and model when needed; errors are re-raised with a more useful first-run message.
- `llm.describe()` reports the configured model details.
- The tokenizer converts an example sentence into token IDs and then human-readable token strings.

The first load can take a while because model weights must be downloaded and loaded into memory. Subsequent runs may use the local cache.

**What to say (English):** “Qwen is the language model that generates the structured analysis. The embedding model and the language model are different components: one retrieves relevant passages, while the other interprets them.”

**بالمصري:** خليك فاكر الفرق: MiniLM للبحث الدلالي، وQwen لتحليل النص وتوليد الـ findings.

## 7. Standalone RAG question answering with checked citations

**RAG** means *Retrieval-Augmented Generation*. ResearchPilot retrieves passages first, then gives the relevant passages to Qwen so the answer can be grounded in the document rather than relying only on the model's general training knowledge.

`answer_question()` retrieves up to three chunks and labels them `E1`, `E2`, and `E3` in the prompt. The model must cite those labels. Program code maps each label back to a stored chunk, so the filename and page are sourced from metadata rather than generated by Qwen.

The two cases below are intentionally different: one asks about information in the PDF; the other asks for a manufacturer that the PDF does not identify. If no retrieval result clears the configured threshold, the QA path can abstain without calling Qwen. If the reply has no valid citation, or its sentences do not match the cited evidence sufficiently, the answer is withheld or marked as unsupported.

> The correct behavior is not “always produce an answer.” It is “answer when the evidence supports it; otherwise state the limitation.”

In [ ]:
from app.rag.qa import answer_question, render_markdown

answerable = next(c for c in CASES["cases"] if c["id"] == "efficiency")
unanswerable = next(c for c in CASES["cases"] if c["id"] == "unanswerable-manufacturer")

for case in (answerable, unanswerable):
    result = answer_question(case["question"], retriever, llm, top_k=3)
    display(Markdown(render_markdown(result)))
    print(f"Expected page(s): {case['expected_pages'] or 'none (not in the PDF)'}")
    print(f"Raw model output: {result.raw_output!r}\n" + "-" * 80)

### Cell explanation — compare an answerable and an unanswerable question

- The first case asks about efficiency, which is covered by the synthetic report.
- The second asks for manufacturer information missing from the report.
- Both questions pass through the same `answer_question()` function and retrieval threshold.
- The rendered result displays the answer and citations, while the raw output helps diagnose what Qwen actually returned.
- `expected_pages` comes from the test case and is included as a comparison aid. It is not the model's answer.

Inspect the actual result carefully. If a relevant answer is missing, retrieval or validation may be too strict; if unsupported content is returned, the safeguards may have failed. Both are valuable findings for evaluation.

**What to say (English):** “I test both a question supported by the PDF and one that the PDF cannot answer. This checks whether the system can ground its answer and recognize missing evidence.”

**بالمصري:** مش بنختبر السؤال اللي إجابته موجودة بس؛ بنختبر كمان هل السيستم هيعترف إنه مش لاقي دليل ولا هيخمن.

### Controlled citation-validation test (no AI call)

The next cell uses a **hand-written fixed reply**, not Qwen. This isolates the code's citation handling: `E1` exists and should resolve to its stored passage; `E9` is deliberately invented and should be discarded.

This test proves that the citation-validation path rejects a fabricated source label. It does **not** measure language-model intelligence or overall answer accuracy.

In [ ]:
from app.services.llm import Generation


# Hand-written reply: E1 exists, E9 does not.
class HandWrittenReply:
    model_id = "hand-written reply (not a model)"

    def generate(self, messages, *, max_new_tokens=256):
        text = "ANSWER: The round-trip efficiency was 90.5%.\nEVIDENCE: E1, E9"
        return Generation(text, prompt_tokens=0, completion_tokens=0, duration_ms=0.0)


checked = answer_question(answerable["question"], retriever, HandWrittenReply())
print("Kept:", [(c.label, c.filename, c.page_number) for c in checked.citations])
print("Discarded:", checked.invalid_citations)

### Cell explanation — deliberately test an invented citation

- `HandWrittenReply` implements the same `generate(...)` interface expected by the QA path, but its text is fixed by us.
- It cites both `E1` and non-existent `E9` on purpose.
- `answer_question(...)` runs the usual citation validation against that fixed reply.
- `checked.citations` shows kept citation references and `checked.invalid_citations` shows rejected labels.

Because there is no model call here, this is a unit-style demonstration of a guardrail, not an LLM evaluation.

**What to say (English):** “I inject a fake citation to verify that the code rejects source IDs that were never retrieved.”

**بالمصري:** دي تجربة متحكم فيها: حاطين citation غلط بإيدينا عشان نتأكد إن الكود مش هيثق في أي ID الموديل يكتبه.

## 8. LangGraph orchestration through the actual FastAPI app

The earlier RAG demonstration calls the QA function directly. This section exercises the **real FastAPI application and the real graph**, using FastAPI's `TestClient` so a separate server process is not necessary.

A temporary data directory keeps this notebook run isolated. The already-loaded Qwen instance is injected into the application so the model is not loaded a second time. The health endpoint checks application readiness, while the graph edges make the routing visible. This is a real local API test, not a screenshot or a mocked graph.

In [ ]:
import tempfile

from fastapi.testclient import TestClient

from app.config import Settings
from app.main import create_app

settings = Settings(
    _env_file=None,
    data_dir=Path(tempfile.mkdtemp(prefix="researchpilot-walkthrough-")),
    llm_device=DEVICE,
    log_level="WARNING",  # keep request logs out of the output
)
app = create_app(settings, llm_factory=lambda: llm)
client = TestClient(app)
print("GET /health →", client.get("/health").json())

graph = app.state.research_service.workflow.get_graph()
for edge in graph.edges:
    print(f"  {edge.source:>16} {'⇢' if edge.conditional else '→'} {edge.target}")

### Cell explanation — construct and inspect the real app

- `tempfile.mkdtemp(...)` creates an isolated data location for this demo run.
- `Settings(...)` configures the local test application without loading a `.env` file, and sets the chosen device and log verbosity.
- `create_app(...)` constructs the real FastAPI app; `llm_factory=lambda: llm` reuses the model instance already loaded above.
- `TestClient(app)` lets Python issue HTTP requests to the app's real routes without starting a separate terminal server.
- `GET /health` checks the health route.
- `get_graph()` and `graph.edges` expose the workflow transitions, including conditional edges.

**What to say (English):** “Instead of testing isolated functions only, I create the actual API application and inspect its LangGraph routes through FastAPI's test client.”

**بالمصري:** إحنا هنا بنجرب التطبيق الحقيقي والـ API من جوّه الـ notebook، من غير ما نحتاج نشغّل سيرفر في Terminal تاني.

In [ ]:
with open(SAMPLE_PDF, "rb") as handle:
    upload = client.post("/documents", files={"file": (SAMPLE_PDF.name, handle, "application/pdf")})
upload.raise_for_status()
DOCUMENT_ID = upload.json()["document_id"]  # taken from the response, never hard-coded
print(
    f"POST /documents → {upload.status_code}: {DOCUMENT_ID}, {upload.json()['chunk_count']} chunks"
)

### Cell explanation — upload the sample PDF through the API

- The PDF is opened in binary mode because file uploads send bytes.
- `client.post("/documents", files=...)` uses the application's real document-upload endpoint.
- `raise_for_status()` stops the notebook if the API returns an HTTP error.
- `DOCUMENT_ID` and `chunk_count` are taken from the API response, avoiding hard-coded identifiers.

This verifies a multi-step part of the workflow: the application accepts a PDF and returns an ID that later requests can reference.

**What to say (English):** “The document enters through the same upload endpoint the application uses, and I reuse the returned document ID in the research request.”

**بالمصري:** رفع الملف مش مجرد قراءة محلية؛ بنبعت الملف للـ endpoint الحقيقي وبناخد الـ ID اللي التطبيق رجّعه.

### What each role does: Analyst, Verifier, and Report Writer

| Stage | Main responsibility | What makes it safer / clearer |
|---|---|---|
| **Planner** | Turns the request into a deterministic plan and its search targets. | Python logic makes the plan inspectable; it is not an LLM-generated plan in this implementation. |
| **Search + document retrieval** | Collects candidate web/demo records and passages from selected PDFs. | Source records retain type, IDs, and document/page metadata. |
| **Analyst (Qwen)** | Turns retrieved passages into structured JSON findings and source IDs. | Findings are proposals; the Analyst's words are not automatically treated as verified facts. |
| **Verifier** | Checks source IDs and metadata, numbers/wording, subject, relevance, and possible prompt-injection issues in code; it can also ask Qwen to review claim support. | A claim is only `verified` when the implementation's required checks pass. These heuristics reduce errors but do not prove truth. |
| **Report Writer** | Produces the Markdown report using the run and verification results. | It is ordinary code, not another free-form LLM answer; a final recommendation is withheld when a required criterion has no verified finding. |

The roles are coordinated by LangGraph. Search and document retrieval can be parallel; the rest of the workflow follows the graph's conditional routes. A rejected finding can trigger one bounded revision, not an unlimited retry loop.

### End-to-end demo: a question with relevant evidence

The request is loaded from the same sample case used in the UI demonstration. The uploaded PDF's `DOCUMENT_ID` is taken from the API response rather than hard-coded. The request enables analysis, then prints the question, duration, analysis status, LLM-call count, and a source table.

Use the source table to answer three questions during the presentation:

1. What evidence did the system collect?
2. Which document and page did that evidence come from?
3. Did the Verifier accept the resulting claim, or explain why it did not?

In [ ]:
request = {**CASES["ui_demo_request"], "document_ids": [DOCUMENT_ID], "analyze": True}
started = time.perf_counter()
response = client.post("/research", json=request)
response.raise_for_status()
run = response.json()
print(f"Question: {run['request']['question']}")
print(
    f"POST /research → {response.status_code} in {time.perf_counter() - started:.1f}s; "
    f"analysis: {run['analysis']['status']}; LLM calls: {run['metrics']['llm_calls']}"
)

sources = {s["id"]: s for s in run["sources"]}


def where(source_id):
    s = sources[source_id]
    return (
        f"{s['filename']} p.{s['page_number']}"
        if s["source_type"] == "document"
        else f"{s['source_type']}: {s['url']}"
    )


table(
    ["ID", "Type", "Reference"],
    [(s["id"], s["source_type"], where(s["id"])) for s in run["sources"]],
)

### Cell explanation — send a research request and list the sources

- `CASES["ui_demo_request"]` supplies a known demonstration request; the uploaded document ID is added and `analyze=True` enables the analysis stage.
- `client.post("/research", json=request)` invokes the real research endpoint.
- The timing and `llm_calls` value show observed runtime information for this one run; they are not a general performance benchmark.
- `sources` maps each source ID to its complete record.
- `where(source_id)` formats the source's file/page or URL for display. The page and filename come from stored source metadata.

Read the output in order: question → status/runtime → source inventory. Only then inspect the model's findings and the Verifier's outcome in the following cell.

**What to say (English):** “The research endpoint orchestrates retrieval and analysis, and I can trace each source ID to a PDF page or a web/demo record.”

**بالمصري:** بنبعت سؤال بحث كامل، ونشوف المصادر اللي اتجمعت ومدة التنفيذ وعدد استدعاءات الـ LLM، وبعدها نراجع كل claim.

In [ ]:
for attempt in run["analysis"]["attempts"]:
    print(f"Analyst attempt {attempt['attempt']}:\n{attempt['raw_output']}\n")

table(
    ["Status", "Claim", "Cites", "Why"],
    [
        (
            c["status"],
            c["claim"],
            "; ".join(f"{sid} ({where(sid)})" for sid in c["cited_source_ids"]),
            " ".join(c["explanations"]),
        )
        for c in run["analysis"]["verification"]["claims"]
    ],
)

### Cell explanation — inspect raw analysis and verification decisions

- The first loop prints each Analyst attempt's raw JSON/text exactly as returned, including a possible bounded revision.
- The table then displays each claim's status, wording, cited source IDs, and the Verifier's explanations.
- `where(sid)` resolves IDs back to the source inventory from the previous cell.
- Compare the model's proposed claim with the cited passage and the final status. An `unverified` or `rejected` result is not a system crash; it is a signal that evidence or validation was insufficient.

**What to say (English):** “The Analyst proposes claims, but the Verifier decides which claims are safe to present as verified. I can inspect the supporting IDs and the reason for each decision.”

**بالمصري:** دور الـ Analyst يقترح المعلومة، لكن مش هو اللي بيدي لنفسه ختم verified. الـ Verifier بيراجع الـ source IDs والدليل وبيوضح سبب القرار.

### Read the generated report back from storage

The API saves the research run. The next cell performs a separate `GET /research/{run_id}` request and checks whether the stored run matches the response returned by `POST /research`.

The report is rendered from the verification outcomes. This is important: an attractive summary from the Analyst is not enough to establish a recommendation. The report should list verified findings, expose missing criteria, and avoid turning missing evidence into a positive conclusion.

In [ ]:
stored = client.get(f"/research/{run['run_id']}").json()
print("Stored run identical to the response:", stored == run)
recommendation = stored["report"].split("## Recommendation")[1].split("## ")[0].strip()
print("Recommendation section:\n" + recommendation)
display(Markdown(stored["report"]))

### Cell explanation — confirm persistence and inspect the final report

- `GET /research/{run_id}` reads the run back from storage using its unique run ID.
- The equality check compares the saved run with the original API response.
- The recommendation is extracted only to make that section easy to inspect in the notebook.
- `display(Markdown(...))` renders the complete report, including findings, criteria coverage, sources, and verification statuses.

In this notebook the app is configured with a temporary directory to keep the test isolated. It demonstrates save-and-read behavior, but it is not intended to preserve the run permanently after the temporary test environment is discarded.

**What to say (English):** “The output is saved through the API and can be fetched again by run ID. The report is derived from the verification results, not just copied from the model's recommendation.”

**بالمصري:** بنثبت إن التقرير اتحفظ ونقدر نجيبه تاني عن طريق run ID. وفي نفس الوقت لازم نفتكر إن فولدر التجربة هنا مؤقت.

## 9. An intentionally unanswerable question: abstention

The sample PDF contains no evidence about electric vehicles. This test is intended to show that a retrieved but unrelated sentence must not be accepted simply because it came from the PDF.

The desired outcome is `insufficient_evidence` and no verified findings supporting the unrelated question. A real model run can still behave imperfectly; inspect the printed status and claim explanations rather than assuming a test passed just because the cell finished. This is one example of a failure-mode test, not proof that every unsupported question will be rejected.

In [ ]:
ev = next(c for c in CASES["cases"] if c["id"] == "unanswerable-ev")
response = client.post(
    "/research", json={"question": ev["question"], "document_ids": [DOCUMENT_ID], "analyze": True}
)
response.raise_for_status()
abstain = response.json()
print(f"Question: {abstain['request']['question']}\nanalysis: {abstain['analysis']['status']}")
for claim in (abstain["analysis"]["verification"] or {}).get("claims", []):
    print(f"  [{claim['status']}] {claim['claim']}  ← {claim['explanations'][-1]}")
summary = abstain["report"].split("## Executive summary")[1].split("## ")[0].strip()
display(Markdown("**Executive summary:** " + summary))

### Cell explanation — test the evidence boundary

- The test case asks about electric vehicles, which are outside the sample report's subject.
- The question goes through the actual `/research` endpoint with the uploaded document ID.
- The cell prints the final analysis status and any verification decisions.
- It extracts the executive-summary section to show how the report communicates missing support.

The intended result is abstention (`insufficient_evidence`) rather than a recommendation based on unrelated text. Always check the live output; never describe expected behavior as a successful observed result if the cell shows otherwise.

**What to say (English):** “This negative test checks whether the system refuses to make a decision when the document does not contain relevant evidence.”

**بالمصري:** ده اختبار الـ abstention: هل النظام هيقول مفيش دليل كفاية، ولا هيربط معلومة مالهاش علاقة بالسؤال؟

## 10. Automated tests and code quality

The next cell runs two local checks:

- `pytest`: exercises application logic such as graph routing, schemas, citation and injection checks, bounded revisions, API behavior, persistence, and UI behavior.
- `ruff check .`: checks Python files for configured lint issues.

For the automated test suite, Qwen is replaced with scripted responses so tests are repeatable and fast. That means passing tests support the **software logic and contracts**, but do not establish real-model accuracy. The live-model examples in Sections 7–9 are separate demonstrations, and this notebook does not claim to be a complete statistical benchmark of answer quality.

In [ ]:
import re
import subprocess

for command in (["pytest", "-q", "--color=no", "-p", "no:cacheprovider"], ["ruff", "check", "."]):
    completed = subprocess.run(
        [sys.executable, "-m", *command],
        cwd=PROJECT_ROOT,
        capture_output=True,
        text=True,
        env={**os.environ, "NO_COLOR": "1"},
    )
    output = re.sub(r"\x1b\[[0-9;]*m", "", completed.stdout + completed.stderr).strip().splitlines()
    print(f"$ {' '.join(command)}  →  {output[-1]}  (exit code {completed.returncode})")

### Cell explanation — execute tests and linting

- The loop runs `python -m pytest ...` and then `python -m ruff check .` in the project root.
- Output is captured so the notebook stays readable; ANSI color codes are removed.
- Only the last output line, plus the process exit code, is shown. Exit code `0` means that command succeeded; a non-zero code means it needs investigation.

For a final submission, retain the observed output in your own run or record the actual result in the presentation notes. Do not claim a pass based only on the presence of this cell.

**What to say (English):** “Pytest checks the software behavior with controlled model replies, while Ruff checks code style and common lint issues. These are complementary to, not a replacement for, a real-model quality evaluation.”

**بالمصري:** pytest بيختبر منطق التطبيق، وRuff بيراجع الكود. نجاحهم مش معناه إن كل إجابة من الموديل صحيحة.

## 11. Limitations and honest evaluation

- **`verified` is a heuristic, not mathematical proof.** Numeric or wording overlap can miss a changed meaning, such as confusing a rated efficiency of 92% with a measured value of 90.5%.
- A correct paraphrase may still be labelled `unverified`, and a relevant page may fall outside the top-k retrieval window.
- Qwen 1.5B is relatively small. It is practical to run locally, but its reasoning and structured-output quality are limited.
- Demo search records are synthetic unless the backend is explicitly configured for live search. An uploaded-PDF-only run can use genuine extracted text from that PDF, but should not be mixed up with synthetic web evidence.
- Live Tavily search, Supabase pgvector, and Kaggle were not tested in this release; this workflow uses the local PDF/FAISS path. There is no OCR for scanned PDFs.
- Automated tests use scripted LLM replies. They validate code paths, not model accuracy. A larger evaluation set would be needed to report robust accuracy, citation precision/recall, latency distributions, and performance across different documents.

## 12. Questions to prepare for

**Why Qwen 1.5B?** It can run locally with a manageable memory footprint. The trade-off is weaker reasoning than larger models, so the system adds evidence checks and must still disclose errors and limitations.

**What is RAG?** Retrieval-Augmented Generation: find relevant passages first, then generate an answer from those passages.

**What do embeddings do?** They represent text as vectors so semantically similar passages can be found even when the exact words differ.

**Why FAISS?** It provides efficient vector similarity search over the indexed chunks. It is the local vector-search component, not the LLM.

**Why LangGraph?** It explicitly represents stages, conditional routes, and a bounded revision loop, making the workflow easier to inspect and test.

**How are citations checked?** Source labels must map to sources in the current run; document references are resolved through stored metadata. Code checks and review assess whether claims are supported, but the checks are imperfect.

**Why not answer every question?** A confident guess is worse than clearly reporting insufficient evidence. The project treats abstention as a valid result.

**Does passing pytest prove AI accuracy?** No. The suite uses scripted replies to validate application behavior. Model quality needs a separate benchmark with known answers and human review.

In [ ]:
client.close()

### Cleanup

`client.close()` closes the FastAPI test client's resources at the end of the notebook. This is ordinary cleanup; it does not stop a production server or delete the actual project files.

## Final takeaway

ResearchPilot demonstrates a multi-stage research workflow, not just a single prompt to a chatbot. It connects PDF ingestion, semantic retrieval, local LLM analysis, source-aware verification, conditional orchestration, and report generation. Its main design principle is to keep claims tied to evidence and to expose gaps when the evidence is not enough.

The most important limitation to state clearly is that **verification reduces risk; it does not eliminate hallucinations or prove every claim true**.

## 13. Final-project requirements: honest coverage map

The instructor's brief asks for data retrieval, LLM-based processing/summarization, automated outputs, workflow orchestration, a face-camera explanation, and reliability/accuracy/efficiency testing. The table below maps those requirements to what this project **currently demonstrates**.

| Requirement | ResearchPilot implementation shown here | Scope / honest limitation |
|---|---|---|
| **1. Data retrieval** | Page-aware PDF extraction, chunking, MiniLM embeddings, FAISS search, source inventory; web-search stage exists in the workflow. | Demo-mode search may be synthetic. Live Tavily behavior was not tested in this release. No OCR for scanned PDFs. |
| **2. Processing and summarization** | Qwen analyzes retrieved passages into structured findings; the report formats verified findings and evidence gaps. | Small local model; wording and reasoning can fail. |
| **3. Automated outputs** | The API generates a Markdown report and stores the run for retrieval by run ID. | Email sending and external dashboard updates are not implemented in this workflow; do not claim them. |
| **4. Workflow orchestration** | LangGraph coordinates planning, parallel retrieval, evidence assessment, analysis, verification, and report generation, with conditional exits and one bounded revision. | The Planner is deterministic Python rather than a separate LLM planner. |
| **5. Presentation and explanation** | This notebook explains the AI path and provides a demo sequence and sample viva questions. | Record the actual application running on your machine and explain observed outputs. |
| **6. Testing and evaluation** | Pytest/Ruff checks, an answerable RAG example, a fixed invalid-citation test, an unsupported-question test, and measured timing for a sample run. | The unit suite uses scripted model replies; the notebook is not a statistically complete model-accuracy benchmark. |

This alignment is important for a credible final project: describe the implemented outputs accurately and present future integrations as future work, not as existing features.

## 14. Face-camera demo plan (about 4–6 minutes)

### Suggested order

1. **Introduce the problem (20–30 seconds).** Explain that the goal is not just to answer a question, but to produce evidence-backed research findings and avoid unsupported recommendations.
2. **Show the architecture (30–45 seconds).** Point to retrieval, Analyst, Verifier, and Report Writer. Explain that LangGraph orchestrates the stages.
3. **Demonstrate a real PDF upload (45–60 seconds).** Show the uploaded document and explain the page-aware extraction and retrieval.
4. **Ask an answerable question (60–90 seconds).** Show the sources, the Analyst's structured output, and the Verifier's decision. Open a source passage and relate it to the claim.
5. **Show the generated report (30–45 seconds).** Point out verified claims, missing criteria, citations, and the recommendation policy.
6. **Ask an unanswerable question (30–45 seconds).** Show the intended `insufficient_evidence` behavior if it occurs. If it does not, honestly explain the failure and what must improve.
7. **Show tests and limitations (30–45 seconds).** Explain the role of pytest/Ruff and be clear that scripted unit tests do not prove the model is accurate.

### A natural English introduction

“Hi, this is ResearchPilot, my final project for building a multi-agent AI system. The goal is to help a user investigate a question using evidence from uploaded PDFs. The system retrieves relevant passages, asks a local language model to propose structured findings, checks the claims and citations, and creates a report. An important design choice is that it should show when the evidence is not sufficient instead of making an unsupported recommendation.”

### What to emphasize

- The key difference from a basic chatbot is the **multi-step workflow and verification layer**.
- The source ID is mapped back to stored source metadata; the model does not get to invent the filename or page number.
- `verified` means the configured checks passed; it does not mean mathematical certainty.
- Real outputs can vary. Describe what your run actually shows.

## 15. Oral-defense questions and quick answers

**Q: What is the role of the LLM?**  
A: Qwen interprets retrieved passages and proposes structured research findings. The application code handles routing, metadata, validation, and report construction.

**Q: What is the difference between embeddings and Qwen?**  
A: Embeddings convert text into vectors for semantic retrieval. Qwen is the generative language model that analyzes the selected text.

**Q: Why not send the whole PDF directly to Qwen?**  
A: Retrieving relevant chunks focuses the context, reduces irrelevant text, and preserves page references for citations.

**Q: Why does the project use chunk overlap?**  
A: Overlap helps keep context when a sentence or explanation falls near a chunk boundary.

**Q: What does FAISS do?**  
A: It indexes vectors and retrieves nearby vectors; it does not generate answers or verify facts.

**Q: What makes this a multi-agent workflow?**  
A: Different stages have clear responsibilities and pass structured state through a LangGraph workflow. The Analyst and Reviewer use the LLM, while other stages are deterministic retrieval or code stages. It is more accurate to describe it as a role-based multi-stage agent system than to claim every stage is an independent autonomous LLM.

**Q: How are claims verified?**  
A: Code checks that cited source IDs exist and checks relevant metadata and textual signals; an LLM-assisted review may also assess support. Both can make mistakes, so the report shows status and explanation.

**Q: What happens when the answer is missing from the PDF?**  
A: The intended behavior is to return an evidence-insufficient status and avoid a final recommendation that is not supported by verified findings.

**Q: Are you using live web search?**  
A: This walkthrough uses Demo mode for search. Those sample results are synthetic unless the backend is explicitly configured and tested for live search. The PDF passages shown here come from the uploaded PDF text.

**Q: Does the project send emails or update an external dashboard?**  
A: Not in the demonstrated implementation. It generates and stores a Markdown research report. Email and dashboard integrations would be future extensions.

**Q: Does passing the tests prove the answers are correct?**  
A: No. The scripted tests validate software behavior. Evaluating answer quality requires a benchmark set with known answers and human-reviewed evidence support.

**Q: What would you improve next?**  
A: Improve retrieval for multi-part questions, calibrate the verification logic to reduce false rejections, add a repeatable evaluation benchmark, test live web integration, and add OCR only if scanned PDFs are an explicit requirement.

## 16. Final run checklist

Before recording or submitting, check each item:

- [ ] Opened this notebook from the **updated** ResearchPilot repository and selected `.venv`.
- [ ] Ran setup/dependency checks successfully.
- [ ] Loaded Qwen successfully on the selected device.
- [ ] Confirmed the PDF used in the demo is the intended sample file and identified it as synthetic test data.
- [ ] Inspected retrieved chunks, source IDs, page metadata, and similarity scores.
- [ ] Ran both the answerable and unanswerable question cases and reported the actual statuses honestly.
- [ ] Inspected the Analyst output next to the Verifier decisions.
- [ ] Confirmed the report can be fetched by run ID.
- [ ] Ran pytest and Ruff and noted their actual exit codes.
- [ ] Can explain that the verifier reduces risk but is not proof, and that scripted tests are not an LLM accuracy benchmark.
- [ ] The screen recording shows the app in action and includes face-cam explanation as required.

**Final principle:** show the evidence, show the workflow, show what passed or failed, and be honest about the current limits.